# P4CYGL processing

This Plasmidsaurus order was originally processed locally (Natascha's laptop). Excessive read duplication revealed a problem with the received files. Below is the reprocessing after Plasmidsaurus made changes to the files.

Reprocessing is started using AISynbioPipeline code on 12-10-25 on Poplar.

## Steps
1. Download, data organization, file renaming
2. Creating seqsamples and cross-checking in LIMS.
3. Short reads: QA/QC
4. (IN PROGRESS) Short reads: Breseq (refseq: ACN2586) and Breseq analysis
5. (TO DO) Long reads: QA/QC
6. (TO DO) Long reads: Blast analysis

## 0. Set-up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## 1. Download, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and libraries.
- Copy all nanopore fastqs into long lib, renaming in the process.
- Copy all illumina fastqs into short lib, renaming in the process.
Creating sample seqs and cross-checking with LIMS


In [91]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'P4CYGL'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
os.makedirs(home_dir)
print(home_dir)

/storage/nspahr/lib_analysis/Plasmidsaurus_2025-09-30_P4CYGL


In [8]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

ITEM P4CYGL
{'code': 'P4CYGL',
 'done_date': '2025-09-30T21:37:37.602680+00:00',
 'gross': 540.0,
 'order_name': '',
 'product_name': 'hybrid_extraction',
 'quantity': 3,
 'status': 'complete'}



DOWNLOADING READS FOR P4CYGL 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2025-09-30_P4CYGL/P4CYGL_reads.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2025-09-30_P4CYGL/P4CYGL_reads.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2025-09-30_P4CYGL/P4CYGL_reads


In [9]:
os.listdir(reception_dir)

['P4CYGL_reads.zip', 'P4CYGL_reads']

In [16]:
# Spot-check if duplicate read ID problem is fixed

os.listdir(os.path.join(reception_dir, 'P4CYGL_reads'))

['P4CYGL_3_ANLstock.ACN2586.colony3_nanopore.fastq.gz',
 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz',
 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R2.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_nanopore.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_illumina_R1.fastq.gz',
 'P4CYGL_1_ANLstock.ACN2586.colony1_illumina_R2.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_nanopore.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_illumina_R1.fastq.gz',
 'P4CYGL_2_ANLstock.ACN2586.colony2_illumina_R2.fastq.gz']

In [14]:
!gunzip -c {os.path.join(reception_dir, 'P4CYGL_reads', 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz')} | head

@LH00941:53:22CCH7LT1:1:1101:1351:1128 1:N:0:AGTCAGAC+TGTCGCTG
TTNAACAGCATTATGGCATAAGCGCCGTAATAGCCGTGTGTTTATGAATAAAAGCGTCGCTGAGATTAGCGAGATTCTGTTTAAAGAATGGCAAAATAGAAGTAGTTTATTTGCAGCGAGTTTAACGCTTGATTTAAGCGGACTTGCTCAA
+
II#IIIIIIIIIIIIIIIIIII9II9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIII9IIII9IIIIII
@LH00941:53:22CCH7LT1:1:1101:1407:1128 1:N:0:AGTCAGAC+TGTCGCTG
CANTATTCTCTGAATATAATGCTTTAAGATTGCCATTTATTCAGGCCTTAGTTACTCATTCAAAGCATCTTAGTGCTAACTATGTTAATTATGCAGATTCGAACTAATTTTTATTATATATAGTTCAAGAAAAGAATGAAATTATTAACTA
+
II#IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9II9IIIIIIIIIIIIIIIIIII
@LH00941:53:22CCH7LT1:1:1101:1740:1128 1:N:0:AGTCAGAC+TGTCGCTG
GTNGTGTATCTCGTGCGTTTAGTCAGCTTGCAGTGGGTGACATTATCGAGATTTCACAACCACAAGGTGGCTTTTATCTTGAACCTAAAGCAGATGGCATTGTCCTGATTGCATCTGGAAGCGGAATAACGGCCATCTATGCTTTGCTACA

gzip: stdout: Broken pipe


In [15]:
!gunzip -c {os.path.join(reception_dir, 'P4CYGL_reads', 'P4CYGL_3_ANLstock.ACN2586.colony3_illumina_R1.fastq.gz')} | grep "@LH00941:53:22CCH7LT1:1:1101:1351:1128 1:N:0:AGTCAGAC+TGTCGCTG"

@LH00941:53:22CCH7LT1:1:1101:1351:1128 1:N:0:AGTCAGAC+TGTCGCTG


First read ID only occurs once. I assume that the duplication problem is fixed.

In [46]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
long = Library(seqorder, 'Nanopore', create=True)

In [48]:
# Identify Illumina/ Nanopore reads and copy into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
i_pattern = "*illumina*.fastq.gz"
n_pattern = "*nanopore*.fastq.gz"
i_files = list(reads_path.glob(i_pattern))
n_files = list(reads_path.glob(n_pattern))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for file in i_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for file in n_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, long.path/'received'/aisynbio_basename)

In [55]:
aisynbio_basename

'ANLstock.ACN2586.colony2_nanopore.fastq.gz'

## 2. Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [66]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)
%run util_simple.py

✓ LIMS API loaded successfully


In [87]:
# Querying LIMS for desired sequencing measurements

thisExpLIMSseqsamples_short = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Short_DNA_reads'}
)['Name'].apply(lambda x: x.replace('.Short_DNA_reads', '')).to_list()

thisExpLIMSseqsamples_long = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Long_DNA_reads'}
)['Name'].apply(lambda x: x.replace('.Long_DNA_reads', '')).to_list()

In [88]:
# Cross-checking seq sample measurement names

short_manifest = short.create_manifest('received')
print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name'].to_list()]))
print(f"Are all short LIMS seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in short_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_short]))

long_manifest = long.create_manifest('received')
print(f"Are all Plasmidsaurus long seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_long for x in long_manifest['sample_name'].to_list()]))
print(f"Are all LIMS long seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in long_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_long]))

Are all short Plasmidsaurus seqsamples from order P4CYGL in the LIMS?
True
Are all short LIMS seqsamples with selected filters in Plasmidsaurus order P4CYGL?
False
Are all Plasmidsaurus long seqsamples from order P4CYGL in the LIMS?
True
Are all LIMS long seqsamples with selected filters in Plasmidsaurus order P4CYGL?
False


In [89]:
# Creating batch (list) of seqsamples for this seqorder

shortSeqSamples = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iterrows()]

## 3. Short reads: QA/QC

- fastp (Celery): Must start running workers first
- MultiQC

In [120]:
# Create trimmed dir

short.create_subfolder('trimmed')

Running one fastp worker:


(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1
======================================================================
Starting fastp Celery Worker
======================================================================

Monitor at: http://poplar.cels.anl.gov:5555
======================================================================

 
 -------------- fastp_1@seed.jupyter v5.6.0 (recovery)
--- ***** ----- 
-- ******* ---- Linux-6.11.0-21-generic-x86_64-with-glibc2.39 2025-12-11 20:22:39
- *** --- * --- 
- ** ---------- [config]
- ** ---------- .> app:         fastp:0x71bd709f79e0
- ** ---------- .> transport:   redis://bioseed_redis:6379/10
- ** ---------- .> results:     redis://bioseed_redis:6379/10
- *** --- * --- .> concurrency: 2 (prefork)
-- ******* ---- .> task events: OFF (enable -E to monitor tasks in this worker)
--- ***** ----- 
 -------------- [queues]
                .> fastp            exchange=fastp(direct) key=fastp

In [109]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

In [110]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [122]:
# Submit tasks:

results = []

for sample in seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [126]:
for i in results:
    print(i.status)

SUCCESS
SUCCESS
SUCCESS


In [134]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, 'trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)


/// ]8;id=110791;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.33 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2025-09-30_P4CYGL/Plasmidsaurus_2025-09-30_P4CYGL_Illumina/trimmed
             fastp | Found 3 reports


        searching | ████████████████████████████████████████ 100% 13/13                                                   html

     write_results | Data        : multiqc_data   (overwritten)
     write_results | Report      : multiqc_report.html   (overwritten)
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_2025-09-30_P4CYGL/trimmed_multiqc_report.html'

## 4. Short reads: Breseq (refseq: ACN2586)

- Run breseq on all illumina seqsamples.
- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.
- 

Started two breseq workers:

aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
======================================================================
Starting breseq Celery Worker
======================================================================

Monitor at: http://poplar.cels.anl.gov:5555
======================================================================

 
 -------------- breseq_1@seed.jupyter v5.6.0 (recovery)
--- ***** ----- 
-- ******* ---- Linux-6.11.0-21-generic-x86_64-with-glibc2.39 2025-12-09 23:29:03
- *** --- * --- 
- ** ---------- [config]
- ** ---------- .> app:         breseq:0x7f7c139263c0
- ** ---------- .> transport:   redis://bioseed_redis:6379/10
- ** ---------- .> results:     redis://bioseed_redis:6379/10
- *** --- * --- .> concurrency: 2 (prefork)
-- ******* ---- .> task events: OFF (enable -E to monitor tasks in this worker)
--- ***** ----- 
 -------------- [queues]
                .> breseq           exchange=breseq(direct) key=breseq



(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2
======================================================================
Starting breseq Celery Worker
======================================================================

Monitor at: http://poplar.cels.anl.gov:5555
======================================================================

 
 -------------- breseq_2@seed.jupyter v5.6.0 (recovery)
--- ***** ----- 
-- ******* ---- Linux-6.11.0-21-generic-x86_64-with-glibc2.39 2025-12-09 23:29:11
- *** --- * --- 
- ** ---------- [config]
- ** ---------- .> app:         breseq:0x7a8f4dd577a0
- ** ---------- .> transport:   redis://bioseed_redis:6379/10
- ** ---------- .> results:     redis://bioseed_redis:6379/10
- *** --- * --- .> concurrency: 2 (prefork)
-- ******* ---- .> task events: OFF (enable -E to monitor tasks in this worker)
--- ***** ----- 
 -------------- [queues]
                .> breseq           exchange=breseq(direct) key=breseq

In [217]:
# Create breseq dir

short.create_subfolder('breseq')

In [139]:
# List available reference genomes

from aisynbiopipeline.workflows.breseq import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk', 'ACN2821_CDM.gbk', 'ACN2586_NSS.gbk']

In [218]:
# Where should this code go?

# Specifies and assigns breseq parameters

def get_breseq_params(seqsample):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': 'ACN2586_NSS.gbk',
        'polymorphism_prediction': True,
        'limit_fold_coverage': 300,
        'num_processors': 4
    }
    return breseq_params

In [219]:
# Submit tasks

results = []
for sample in shortSeqSamples:
    result = client.send_task(
        'breseq.run',
        kwargs=get_breseq_params(sample),
        queue='breseq'
    )
    results.append(result)

In [223]:
for i in results:
    print(i.status)

STARTED
STARTED
STARTED


In [221]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'TFMN1\.(sohB.pgi|fba.pgi|fba.sohB|sohB.tpiA|fba.tpiA|pgi.tpiA|fba|tpiA|sohB|noDNA|pgi)\.([1-5])\.T(\d{1,2})\.([PSL])([123])?$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        construct = match.group(1)
        replicate = match.group(2)
        transfer = match.group(3)
        isolate = match.group(4)
        colony_num = match.group(5)

    else:
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'
        isolate = 'NA'
        colony_num = 'NA'

    return {'construct': construct, 'replicate': replicate, 'transfer': transfer, 'isolate': isolate, 'colony_num': colony_num}
        

In [222]:
# Create breseq objects for each seqsample and aggregate summary counts

breseq_objects = []

for i in results:
    breseq_folder = i.result['output']
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

rows = []

for b in breseq_objects:
    
    row = {}
    
    try:
        b.count_reads()
        b.count_mutations()
    except Exception as e:
        row.update({'seqsample': getattr(b, 'title', None)})
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row.update({'error': str(e),
                    'input_read_count': None,
                    'used_read_count': None,
                    'mapped_read_count': None,
                    'consensus_mutation_count': None,
                    'polymorphism_mutation_count': None,}
                    )
        rows.append(row)
        print(f"Error loading Breseq from {b.title}: {e}")
        continue

    row['seqsample'] = getattr(b, 'title', None)
    row.update(parse_seqsample_name(getattr(b, 'title', None)))
    row['error'] = None
    row['input_read_count'] = getattr(b, 'input_read_count', None)
    row['used_read_count'] = getattr(b, 'used_read_count', None)
    row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
    row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
    row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)

    rows.append(row)

df = pd.DataFrame(rows)
df

KeyError: 'output'

In [ ]:
from aisynbiopipeline.workflows.breseq import compare_gdiff

reference = 'ACN2586_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

table_format = 'html'
outfile = os.path.join(home_dir, f'all_mutations_{item_code}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(home_dir, f'all_mutations_{item_code}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

None of below has been run. Copied from create_breseq_manifest testing (untitled) notebook. All these transformations should possibly go into a different script...

In [ ]:
# For the same mutation, these fields will be different between the seq samples, so must eliminate in order to do the pivot.
compare_df = compare_df.drop(
    ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies'
    ], axis=1
) 

# Other cols to drop
compare_df = compare_df.drop(['clone', 'mutator_status', 'population', 'time', 'treatment'], axis=1) 

Next, reshape (pivot) the table by grouping by these columns, so that only title (sample) remain as columns (values are the mutation frequencies).
Reindex.
Now ready to analyze

In [ ]:
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

df =compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)

In [ ]:
df

In [ ]:
df.loc[(df == 1).any(axis=1)]

In [ ]:
df.loc[(df == 1).all(axis=1)]

Next, format df like html table, then write out to csv and view in google sheets.

In [ ]:
# Decided against formatting like the html table. Will show to group and let them decide which of the cols are important

df.to_csv(os.path.join(home_lib_path, 'mutation_comparison_P4CYGL_ADP1_Neidle_CDM_reformatted.csv'))

Because there might be many many samples in this comparison, define sample set.

In [ ]:
samples = ['ANLstock.ACN2586.colony1', 'ANLstock.ACN2586.colony2', 'ANLstock.ACN2586.colony3']
sample_order = list(range(3))
sample_set = dict(zip(sample_order, samples))
sample_set

In [ ]:
df_set = df[list(sample_set.values())]

Show fixed mutations present in all samples of sample set

In [ ]:
df[(df == 1).all(axis=1)].reset_index()[['position', 'gene_name', 'mutation_category'] + samples]

Show polymorphisms present in all samples of sample set.

In [ ]:
df[(df!=0).all(axis=1)].reset_index()[['position', 'gene_name', 'mutation_category'] + samples]

Show polymorphisms present in increasing frequency with sample order.

In [ ]:
# From chatGPT. Don't understand why this works. Figure this out!!

df[
    # 1. Row-wise non-decreasing
    (df.values[:, 1:] >= df.values[:, :-1]).all(axis=1)
    &
    # 2. First and last cannot both be min or both be max
    ~(
        ((df.iloc[:, 0] == df.min(axis=1)) & (df.iloc[:, -1] == df.min(axis=1)))
        |
        ((df.iloc[:, 0] == df.max(axis=1)) & (df.iloc[:, -1] == df.max(axis=1)))
    )
]